In [1]:
import torch
import numpy as np
from PIL import Image
import torchvision.transforms.functional as F

%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
# Transforming the image

In [3]:
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, mask, bboxes):
        for t in self.transforms:
            image, mask, bboxes = t(image, mask, bboxes)
        return image, mask, bboxes

In [4]:
class Resizer:
    def __init__(self, output_size=(640, 640)):
        self.output_size = output_size

    def __call__(self, image, mask, bboxes):
        # getting original sizes
        orig_w, orig_h = image.size
        new_w, new_h = self.output_size

        # resize mask and image
        # using bilinear for 4 nearest pixels in original image
        image = F.resize(image, self.output_size, interpolation=Image.BILINEAR)
        # using nearest to find the nearest pixel in the original
        # so the edges of the road are smooth
        mask = F.resize(mask, self.output_size, interpolation=Image.NEAREST)

        rescaled_bboxes = [] 
        ratio_w = new_w/orig_w
        ratio_h = new_h/orig_h

        # resizing the bounding boxes
        for bbox in bboxes:
            x, y, w, h = bbox
            new_x = x * ratio_w
            new_y = y * ratio_h
            new_w = w * ratio_w
            new_h = h * ratio_h
            rescaled_bboxes.append([new_x, new_y, new_w, new_h])

        return image, mask, rescaled_bboxes

In [5]:
class ToTensor:
    def __call__(self, image, mask, bboxes):
        # converting image to float tensor - 0 to 1
        image = F.to_tensor(image)

        # convert mask to long tensor (for segmentation classes)
        # mask is 0 or 255, we convert it to 0 or 1 (is road or isn't)
        mask = torch.from_numpy(np.array(mask)).long()
        mask[mask > 0] = 1

        # bboxes to float tensors
        # if there are no cars, we return an empty tensor with shape (0, 4)
        if len(bboxes) > 0:
            bboxes = torch.as_tensor(bboxes, dtype=torch.float32)
        else:
            bboxes = torch.empty((0, 4), dtype=torch.float32)

        return image, mask, bboxes

In [6]:
# Testing the transformation

In [7]:
import os
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [8]:
def load_single_sample(image_dir, mask_dir, json_path, index=0):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # info for a specific image
    img_info = data['images'][index]
    img_id = img_info['id']
    file_name = img_info['file_name']
    
    # define paths
    img_path = os.path.join(image_dir, file_name)
    # mask has the same name but .png extension
    mask_path = os.path.join(mask_dir, file_name.replace('.jpg', '.png'))
    
    # load using PIL
    image = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L") # grayscale
    
    # extract bboxes for this image ID
    bboxes = [ann['bbox'] for ann in data['annotations'] if ann['image_id'] == img_id]
    
    return image, mask, bboxes

In [9]:
def test_transformation(image_tensor, mask_tensor, bboxes_tensor):
    # convert image tensor [C, H, W] to numpy [H, W, C]
    image = image_tensor.permute(1, 2, 0).numpy()
    
    # convert mask tensor to numpy for plotting
    mask = mask_tensor.numpy()

    # creating one window with two side-by-side plots
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    
    # shows the actual photo
    ax[0].imshow(image)
    ax[0].set_title("Transformed Image & Bboxes")
    
    for bbox in bboxes_tensor:
        x, y, w, h = bbox.numpy()
        # box outline
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
        ax[0].add_patch(rect)
        
    # shows road mask (background - black, road - white)
    ax[1].imshow(mask, cmap='gray')
    ax[1].set_title("Transformed Road Mask")
    
    plt.show()

In [10]:
TRAIN_IMG_DIR = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\train\images'
TRAIN_MASK_DIR = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\train\masks'
TRAIN_JSON = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\train\labels\train_reduced.json'

transform_pipeline = Compose([
    Resizer(output_size=(640, 640)),
    ToTensor()
])

try:
    # load raw data
    raw_img, raw_mask, raw_bboxes = load_single_sample(
        TRAIN_IMG_DIR, 
        TRAIN_MASK_DIR, 
        TRAIN_JSON, 
        index=1
    )

    # apply the transformations
    img_t, mask_t, bbox_t = transform_pipeline(raw_img, raw_mask, raw_bboxes)

    # visualize the results
    test_transformation(img_t, mask_t, bbox_t)

except FileNotFoundError as e:
    print(f"Error: Could not find files. Check your paths!\n{e}")
except Exception as e:
    print(f"An error occurred: {e}")

Error: Could not find files. Check your paths!
[Errno 2] No such file or directory: 'D:\\PODACI SA C\\ROOT\\Projekti\\singi-ml\\SpeedRadarProject\\train\\labels\\train_reduced.json'


In [11]:
# Dataset and DataLoader

In [12]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

In [13]:
class SpeedRadarDataset(Dataset):
    def __init__(self, root_dir, json_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.img_dir = os.path.join(root_dir, "images")
        self.mask_dir = os.path.join(root_dir, "masks")
        
        with open(json_file, 'r') as f:
            self.coco_data = json.load(f)
            
        self.images = self.coco_data['images']
        
        # pre-mapping image IDs to their annotations for optimization
        self.img_id_to_anns = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_id_to_anns:
                self.img_id_to_anns[img_id] = []
            self.img_id_to_anns[img_id].append(ann['bbox'])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']

        # loading image and road mask
        img_path = os.path.join(self.img_dir, file_name)
        mask_path = os.path.join(self.mask_dir, file_name.replace('.jpg', '.png'))
        
        image = Image.open(img_path).convert("RGB")
        road_mask_pil = Image.open(mask_path).convert("L")
        
        bboxes = self.img_id_to_anns.get(img_id, [])
        
        # using transform pipeline
        if self.transform:
            image, road_mask, bboxes = self.transform(image, road_mask_pil, bboxes)
    
        target = {}
        final_boxes = []
        final_labels = []
        final_masks = []

        # class 1 for cars
        for box in bboxes:
            x, y, w, h = box
            # converting to [x1, y1, x2, y2]
            final_boxes.append([x, y, x + w, y + h])
            final_labels.append(1)
            
            # making a placeholder mask since Mask R-CNN requires one for each obj
            car_mask = torch.zeros((640, 640), dtype=torch.uint8)
            car_mask[int(y):int(y+h), int(x):int(x+w)] = 1
            final_masks.append(car_mask)

        # class 2 for roads
        if torch.any(road_mask > 0):
            pos = torch.where(road_mask)
            # box that contains the road
            road_bbox = [torch.min(pos[1]), torch.min(pos[0]), 
                         torch.max(pos[1]), torch.max(pos[0])]
            
            final_boxes.append(road_bbox)
            final_labels.append(2)
            # adding the actual mask of the road
            final_masks.append(road_mask.to(torch.uint8))

        # making into tensors
        if len(final_boxes) > 0:
            target["boxes"] = torch.as_tensor(final_boxes, dtype=torch.float32)
            target["labels"] = torch.as_tensor(final_labels, dtype=torch.int64)
            target["masks"] = torch.stack(final_masks)
        else:
            # case if no road or car
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
            target["labels"] = torch.zeros((0,), dtype=torch.int64)
            target["masks"] = torch.zeros((0, 640, 640), dtype=torch.uint8)

        return image, target

In [14]:
TRAIN_ROOT = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\train'
VAL_ROOT = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\val'

TRAIN_JSON = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\train\labels\train_reduced.json'
VAL_JSON = r'D:\PODACI SA C\ROOT\Projekti\singi-ml\SpeedRadarProject\val\labels\val_reduced.json'

# for handling various box sizes on the same image
def collate_fn(batch):
    return tuple(zip(*batch))

# dataset instances
train_dataset = SpeedRadarDataset(
    root_dir=TRAIN_ROOT, 
    json_file=TRAIN_JSON, 
    transform=transform_pipeline
)

val_dataset = SpeedRadarDataset(
    root_dir=VAL_ROOT, 
    json_file=VAL_JSON, 
    transform=transform_pipeline
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Dataset ready!")
print(f"Batches u train: {len(train_loader)}")
print(f"Batches u val: {len(val_loader)}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\PODACI SA C\\ROOT\\Projekti\\singi-ml\\SpeedRadarProject\\train\\labels\\train_reduced.json'

In [15]:
# Models

In [16]:
import torch.optim as optim
import torch.nn as nn
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

In [17]:
def get_model_instance_segmentation(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # changing the pre-trained head with new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # changing mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # swaping masks with new ones
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask,
                                                       hidden_layer,
                                                       num_classes)

    return model

# num_classes = 3 (0: background, 1: car, 2: road)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model_instance_segmentation(num_classes=3)
model.to(device)

print(f"Model učitan na: {device}")

Model učitan na: cuda


In [18]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [19]:
import math
import sys

def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    header = f"Epoch: [{epoch}]"
    
    all_losses = []
    
    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        loss_value = losses.item()
        if not math.isfinite(loss_value):
            print(f"Loss is {loss_value}, stopping training")
            sys.exit(1)

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        all_losses.append(loss_value)
        
        if i % 10 == 0:
            print(f"{header} Iteration {i}/{len(data_loader)}, Loss: {loss_value:.4f}")

    return sum(all_losses) / len(all_losses)

In [21]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

num_epochs = 10

print(f"Započinjem trening na: {device}")

for epoch in range(num_epochs):
    avg_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    
    lr_scheduler.step()
    
    print(f"--- Završena epoha {epoch}. Prosečan Loss: {avg_loss:.4f} ---\n")

print("Probni trening završen!")

Započinjem trening na: cuda


NameError: name 'train_loader' is not defined

In [22]:
import matplotlib.patches as patches

def visualize_prediction(model, dataset, index=0, device='cuda'):
    model.eval()
    
    image, target = dataset[index]
    
    with torch.no_grad():
        prediction = model([image.to(device)])
    
    img_display = image.permute(1, 2, 0).cpu().numpy()
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img_display)
    
    preds = prediction[0]
    for i in range(len(preds['boxes'])):
        score = preds['scores'][i].item()
        if score > 0.5: # Prikazujemo samo pouzdane detekcije
            box = preds['boxes'][i].cpu().numpy()
            label = preds['labels'][i].item()
            mask = preds['masks'][i, 0].cpu().numpy()
            
            # blue color for car, green for the road
            color = 'blue' if label == 1 else 'green'
            
            rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            
            # drawing mask
            ax.imshow(np.ma.masked_where(mask < 0.5, mask), cmap='jet', alpha=0.3)
            
            ax.text(box[0], box[1], f"{'Auto' if label==1 else 'Put'} {score:.2f}", 
                    color='white', bbox=dict(facecolor=color, alpha=0.5))

    plt.title(f"Predviđanje na slici {index}")
    plt.show()

visualize_prediction(model, val_dataset, index=11, device=device)

NameError: name 'val_dataset' is not defined

In [23]:
import cv2

In [24]:
def get_homography(original_image_pil, model_mask_640):
    orig_w, orig_h = original_image_pil.size
    full_mask = cv2.resize(model_mask_640, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    
    # finding all pixels of the mask
    all_y, all_x = np.where(full_mask > 0.5)
    
    # calculating the range of x and y axis
    delta_x = np.max(all_x) - np.min(all_x)
    delta_y = np.max(all_y) - np.min(all_y)
    
    # if delta_x is significantly larger, we treat it as horizontal
    if delta_x > 1.2 * delta_y:
        # horizontal roads, we take dots on the left and right side
        # moving 10% inside to get a cleaner road shape
        offset_x = int(delta_x * 0.1)
        left_x = np.min(all_x) + offset_x
        right_x = np.max(all_x) - offset_x
        
        # edges of the road are on the top and bottom of that x
        left_indices = np.where(full_mask[:, left_x] > 0.5)[0]
        right_indices = np.where(full_mask[:, right_x] > 0.5)[0]
        
        # ordering points for 400x800 vertical output
        src_pts = np.float32([
            [left_x, left_indices[0]],   
            [right_x, right_indices[0]], 
            [right_x, right_indices[-1]],
            [left_x, left_indices[-1]]   
        ])
    else:
        # vertical road
        # moving inside the mask to avoid catching the curb
        top_y = np.min(all_y) + 30
        bottom_y = np.max(all_y) - 20
        top_indices = np.where(full_mask[top_y, :] > 0.5)[0]
        bottom_indices = np.where(full_mask[bottom_y, :] > 0.5)[0]
        
        src_pts = np.float32([
            [top_indices[0], top_y], 
            [top_indices[-1], top_y],
            [bottom_indices[-1], bottom_y], 
            [bottom_indices[0], bottom_y]
        ])

    # rotating the image to a birds eye view
    dst_pts = np.float32([[0, 0], [400, 0], [400, 800], [0, 800]])
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(np.array(original_image_pil), M, (400, 800))
    
    return warped, M

In [25]:
def extract_line_segments(warped_img):
    # grayscaling the image
    gray = cv2.cvtColor(warped_img, cv2.COLOR_RGB2GRAY)
    # otsu - for extracting lines (sets threshold)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    height, width = thresh.shape
    
    # finding the best x axis by looking for dashed lines
    best_x = width // 2
    max_score = 0
    best_segments = []
    
    # scanning from left to right, staying away from sidewalk edges
    for x in range(50, width - 50, 10):
        # creating the tunnel around current x
        sample_width = 5
        tunnel = thresh[:, max(0, x - sample_width) : min(width, x + sample_width)]
        
        # average of the tunnel to boolean
        line_map = np.mean(tunnel, axis=1) > 127
        
        # grouping into segments
        segments = []
        if len(line_map) > 0:
            count = 1
            for i in range(1, len(line_map)):
                if line_map[i] == line_map[i-1]:
                    count += 1
                else:
                    segments.append((line_map[i-1], count))
                    count = 1
            segments.append((line_map[-1], count))
            
            # scoring system to avoid tram rails or long curbs
            # a real dashed line is not too long and not too short
            transitions = len(segments)
            score = transitions
            
            for is_white, length in segments:
                # if the white line is too long (more than 40% of height), its probably a rail
                if is_white and length > (height * 0.4):
                    score = 0
                    break
            
            # taking the x with best score
            if score > max_score:
                max_score = score
                best_x = x
                best_segments = segments
                
    if not best_segments: 
        return None, None, best_x
        
    # taking the first valid couple
    # finding the true segment (line) and false (gap)
    p_line, p_gap = None, None
    for i in range(len(best_segments) - 1):
        # using a safe range for lines (30px to 300px)
        if best_segments[i][0] == True and 30 < best_segments[i][1] < 300:
            if best_segments[i+1][0] == False and 20 < best_segments[i+1][1] < 400:
                p_line = best_segments[i][1]
                p_gap = best_segments[i+1][1]
                break
                
    # returning p_line, p_gap and the exact x coordinate where we found them
    return p_line, p_gap, best_x

In [26]:
def calculate_meters_per_pixel(p_line, p_gap):
    if p_line is None or p_gap is None:
        print("Sistem nije detektovao jasan par linija-razmak.")
        return None
        
    ratio = p_line / p_gap
    
    # if the line is a lot bigger than the gap, we say its 3m line and 1m gap
    if ratio >= 2.0:
        road_type = "Gradski put (3m linija, 1m razmak)"
        meters_per_pixel = 3.0 / p_line
    else:
        road_type = "Otvoreni put (3m linija, 3m razmak)"
        meters_per_pixel = 3.0 / p_line
        
    # check if the scale is too crazy (safety check)
    if meters_per_pixel > 0.5:
        print("Upozorenje: Scale factor je sumnjivo veliki!")
        
    print(f"Logika: {road_type}")
    print(f"Pikseli: Linija={p_line}px, Razmak={p_gap}px (Odnos: {ratio:.2f})")
    print(f"Finalni Scale: {meters_per_pixel:.5f} metara po pikselu")
    
    return meters_per_pixel

In [27]:
def test_radar_system(test_image, test_mask):
    print("--- Započinjem testiranje sistema ---")
    
    # 1. homography setup (ispravljanje puta)
    warped_img, M = get_homography(test_image, test_mask)
    
    if warped_img is None:
        print("Greška: Homografija nije uspela. Proveri masku puta.")
        return

    # 2. extracting lines and gaps (sada uzimamo i best_x)
    p_line, p_gap, best_x = extract_line_segments(warped_img)
    
    # 3. calculating math (ratio and scale)
    scale = calculate_meters_per_pixel(p_line, p_gap)
    
    # --- VIZUELIZACIJA ---
    plt.figure(figsize=(12, 6))
    
    # original image + mask
    plt.subplot(1, 2, 1)
    plt.imshow(test_image)
    # resizing mask to fit the image
    plt.imshow(cv2.resize(test_mask, (test_image.size[0], test_image.size[1])), alpha=0.3, cmap='jet')
    plt.title("Originalna slika + Maska puta")
    
    # warped image (bird's eye)
    plt.subplot(1, 2, 2)
    plt.imshow(warped_img)
    plt.title("Bird's Eye View (Ispravljen put)")
    
    # drawing the tunnel where the lines were ACTUALLY found
    if best_x is not None:
        plt.axvline(x=best_x, color='r', linestyle='--', label=f'Skenirani tunel (x={best_x})')
        plt.legend()
        
    plt.show()

    # ispis rezultata
    if scale:
        print(f"\nUSPEH!")
        print(f"Rezultat: Svaki piksel na 'warped' slici iznosi {scale:.5f} metara.")
        print(f"To znači da automobil koji pređe 100px zapravo pređe {100 * scale:.2f} metara.")
    else:
        print("\nSistem nije uspeo da izračuna skalu. Nisu pronađene jasne linije.")

In [28]:
def run_random_test_v2(dataset, model, device='cuda'):
    # 1. Uzmi nasumičan primer
    idx = random.randint(0, len(dataset) - 1)
    image, _ = dataset[idx]
    
    model.eval()
    with torch.no_grad():
        prediction = model([image.to(device)])
    
    preds = prediction[0]
    road_mask = None
    highest_score = 0
    
    # 2. Pronađi masku koja je LABEL 2 (PUT) sa najvećim score-om
    for i in range(len(preds['labels'])):
        label = preds['labels'][i].item()
        score = preds['scores'][i].item()
        
        if label == 2 and score > 0.5:
            if score > highest_score:
                highest_score = score
                road_mask = preds['masks'][i, 0].cpu().numpy()
    
    # 3. Ako smo našli put, nastavi sa testom
    if road_mask is not None:
        # Pripremi PIL sliku za tvoje funkcije
        img_np = image.permute(1, 2, 0).cpu().numpy()
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
        pil_image = Image.fromarray(img_np)
        
        # Pozovi ranije napisane funkcije za test sistema
        # One će sada dobiti pravu masku puta!
        test_radar_system(pil_image, road_mask)
    else:
        print(f"Na slici {idx} model nije pronašao put (Label 2) sa score > 0.5")

# Pokreni test ponovo
run_random_test_v2(val_dataset, model, device=device)

NameError: name 'val_dataset' is not defined